<!-- COMMONS LAUNCHER v3 · generated by tools/notebooks.py · do not edit by hand -->
<a href="https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials"><img src="https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/brand/synapsa-commons-badge.png" alt="Synapsa Commons" height="36"></a>

Free, hands-on AI courses that run anywhere, from the team building Synapsa, an AI-native
learning platform.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/programmes/predictive-maintenance/lessons/P03-L06-thresholds-from-distributions/lesson.ipynb)
[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/programmes/predictive-maintenance/lessons/P03-L06-thresholds-from-distributions/lesson.ipynb)
[![Open in Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master?labpath=programmes/predictive-maintenance/lessons/P03-L06-thresholds-from-distributions/lesson.ipynb)
[![Open in Codespaces](https://github.com/codespaces/badge.svg)](https://codespaces.new/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials)

This lesson needs Python 3.11 or newer with numpy and matplotlib, which Colab, Kaggle,
Binder and Codespaces already have.

In [ ]:
# --- COMMONS LAUNCHER v3 · generated by tools/notebooks.py · do not edit by hand ---
# Makes this notebook run anywhere. Every line is a no-op when the thing is already present,
# so a local clone pays nothing and an online notebook repairs itself.
import importlib.util, os, subprocess, sys, urllib.request
from pathlib import Path

COMMONS_PIP = []            # (import name, pinned pip spec) for what this lesson imports
COMMONS_SIBLINGS = []    # files that must sit beside the notebook
# A fork, a classroom mirror or an offline copy can serve the files from elsewhere by setting
# COMMONS_RAW_OVERRIDE before running this cell.
COMMONS_RAW = os.environ.get("COMMONS_RAW_OVERRIDE") or "https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/programmes/predictive-maintenance/lessons/P03-L06-thresholds-from-distributions/"

# Resolve siblings against the LESSON's own directory, not the working directory. A notebook
# has no __file__ and runs with cwd alongside itself; a grader imports this file from the repo
# root. Checking cwd blindly makes the grader think every sibling is missing and reach for the
# network -- which would put a download on a graded path.
try:
    COMMONS_DIR = Path(__file__).resolve().parent
except NameError:
    COMMONS_DIR = Path.cwd()


def commons_host() -> str:
    """Name the notebook service we are on. Used for the message, and for honest errors."""
    try:
        if importlib.util.find_spec("google.colab") is not None:
            return "Google Colab"
    except (ImportError, ValueError):
        pass
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
        return "Kaggle"
    if os.environ.get("BINDER_SERVICE_HOST"):
        return "Binder"
    if os.environ.get("CODESPACES"):
        return "GitHub Codespaces"
    return "a local Python environment"


_missing = [pip for imp, pip in COMMONS_PIP if importlib.util.find_spec(imp) is None]
if _missing:
    print("installing " + ", ".join(_missing) + " ...")
    # pip everywhere a student is likely to be; uv-managed local venvs ship without pip.
    if importlib.util.find_spec("pip") is not None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_missing], check=True)
    else:
        subprocess.run(["uv", "pip", "install", "-q", "--python", sys.executable, *_missing],
                       check=True)
    importlib.invalidate_caches()

_fetched = []
for _name in COMMONS_SIBLINGS:
    if not (COMMONS_DIR / _name).exists():
        (COMMONS_DIR / _name).parent.mkdir(parents=True, exist_ok=True)
        try:
            urllib.request.urlretrieve(COMMONS_RAW + _name, COMMONS_DIR / _name)
            _fetched.append(_name)
        except Exception as _e:  # Kaggle disables the internet by default; say so plainly
            raise RuntimeError(
                f"this lesson needs {_name} beside the notebook and could not fetch it "
                f"({_e}). On Kaggle, switch Internet on in the notebook settings panel "
                f"(Kaggle allows that only for phone-verified accounts); otherwise download it "
                f"from {COMMONS_RAW + _name} and upload it beside the notebook."
            ) from None

print("ready on " + commons_host() + ("; fetched " + ", ".join(_fetched) if _fetched else ""))
# --- END COMMONS LAUNCHER ---

# P03-L06 · Threshold setting from an RUL distribution

**You will build:** the decision layer that sits on top of module 5's remaining-useful-life
forecast. A summary of a predictive RUL distribution, its survival and hazard curves, the
expected cost of committing to intervene at a given hour, the optimal *intervention time*,
the same cost evaluated at the real slots in a workshop calendar, and two schedulers — the
one a planner writes on a whiteboard, and the one that is actually cheapest — for a workshop
that cannot take two machines on the same Thursday.

**Time:** ~80 minutes · **Runs on:** a laptop CPU, 8 GiB RAM, no GPU, no download
· **Prerequisites:** `T00-L01-the-8gb-track` (the profiler and the tier gate),
`P03-L01-alarm-economics` (the three prices, the sweep, and the habit of defending an
operating point in currency). Module 5, `P03-L05-remaining-useful-life`, is the source of
the forecast this module consumes; while it is being built, this notebook generates
forecasts in the same shape itself — see section 1.

By the end you will be able to:

1. Implement the expected cost of intervening at hour *t* given a predictive RUL
   distribution, and locate the cost-optimal intervention time.
2. Measure how far that time is from the mean RUL, on your own numbers.
3. Measure how the optimal intervention time moves when the *width* of the distribution
   changes and its centre does not — and why, from the hazard rate you compute yourself.
4. Price a plan against a maintenance calendar with real slots in it, rather than against a
   continuous clock.
5. Implement a capacity-constrained schedule, and measure what the workshop's one-machine-
   per-slot rule costs against the unconstrained optimum — and what a first-come-first-served
   planner costs on top of that.

**The data is synthetic and the generator is in this notebook.** Nothing is downloaded.
`meta.yaml` declares it. The method transfers; the particular numbers describe no real
plant, and the three prices are a scenario rather than a measurement.

In [ ]:
# Setup: everything the lesson needs, in one cell, with versions printed.
import contextlib
import io
import math
import sys
import time
import traceback
from typing import Callable, NamedTuple, Sequence

import numpy as np

import matplotlib
_INTERACTIVE = "ipykernel" in sys.modules
if not _INTERACTIVE:
    # Headless: a script run (including this repository's execution gate) must never try to
    # open a window. In Jupyter the default inline backend is already the right one.
    matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402  (backend must be chosen before this import)

_LESSON_T0 = time.perf_counter()
print("python", sys.version.split()[0], "· numpy", np.__version__,
      "· matplotlib", matplotlib.__version__)

SEED = 20260922

# "Now". Every forecast in this notebook is made at the same moment, because that is how a
# planner uses one: the model runs overnight and the whole fleet is re-planned in the morning.
NOW = 0

# The RUL grid. Bin k holds the probability that the unit fails during hour k — that is, that
# its remaining useful life lands in (k, k+1]. 900 hours is a little over five weeks, which is
# past the far tail of every forecast here; whatever mass lies beyond it is folded into the
# last bin rather than thrown away.
GRID_HOURS = 900

# The candidate intervention times, in hours ahead of NOW. This is the continuous-clock sweep:
# section 9 replaces it with the slots a workshop actually has.
CANDIDATES = np.arange(0, 601, dtype=int)

# The workshop calendar. It opens at 08:00 and takes machines on weekdays only. Hour 0 is a
# Monday midnight, so slot hours are 8, 32, 56, 80, 104 — and then nothing until 176.
WORKSHOP_OPEN_HOUR = 8
WORKING_DAYS = (0, 1, 2, 3, 4)
CALENDAR_DAYS = 28

# The planner negotiates over the slots in the next fortnight, not over the whole calendar.
SHORTLIST = 8
CAPACITY = 1          # one machine per slot: the constraint sections 11 and 12 are about
N_FLEET = 6


class RULForecast(NamedTuple):
    """Module 5's deliverable for one unit, at one moment.

    `hours` is the RUL grid in hours ahead of `made_at`; `pmf[k]` is the probability that the
    unit fails during hour `hours[k]`. `pmf` sums to 1. This is the whole interface: a
    distribution, not a number, and nothing downstream is allowed to collapse it early.
    """
    unit: int
    made_at: int
    hours: np.ndarray
    pmf: np.ndarray


class Summary(NamedTuple):
    """What module 5 would put on a dashboard. Useful, and not the answer to any question."""
    mean: float
    sd: float
    p10: int
    p50: int
    p90: int


class Prices(NamedTuple):
    """Module 1's cost model, with the false alarm replaced by something continuous.

    Module 1 priced a false alarm as an event: you either cried wolf or you did not. Against a
    distribution there is no such event. Every intervention is early by some amount, and what
    it costs is the life you threw away — `life_per_hour` for every hour of remaining useful
    life still on the machine when you pulled it. A scenario, not a measurement.
    """
    planned: float
    unplanned: float
    life_per_hour: float


class Plan(NamedTuple):
    """One machine's intervention time, in hours ahead of the forecast, and what it costs."""
    hours_ahead: int
    cost: float


class Schedule(NamedTuple):
    """An assignment of machines to calendar slots, and the total expected cost."""
    assignment: tuple
    total: float


SCENARIO = Prices(planned=9_000.0, unplanned=180_000.0, life_per_hour=220.0)

_FAILED_CHECKS: list[str] = []

# The exercises, in the order you meet them, and the functions each one asks you to write.
# The progress board at the foot of the notebook is built from this, and a cell that is
# waiting on an unfinished exercise names it from here.
_EXERCISES: dict[str, tuple[str, ...]] = {
    "exercise 1": ("forecast_summary",),
    "exercise 2": ("survival_and_hazard",),
    "exercise 3": ("expected_cost_at",),
    "exercise 4": ("best_intervention_time",),
    "exercise 5": ("realised_cost",),
    "exercise 6": ("slot_costs",),
    "exercise 7": ("fleet_cost_matrix",),
    "exercise 8": ("greedy_schedule",),
    "exercise 9": ("best_schedule",),
}
_STATUS: dict[str, str] = {}   # label -> "passed" | "failed" | "not started", latest run


def _named(labels: list[str]) -> str:
    """["exercise 3"] -> "exercise 3 (expected_cost_at)"; several -> "exercises 1, 3 and 4"."""
    if len(labels) == 1:
        return f"{labels[0]} ({', '.join(_EXERCISES[labels[0]])})"
    nums = [label.split()[-1] for label in labels]
    return "exercises " + ", ".join(nums[:-1]) + " and " + nums[-1]


def _try(label: str, check: Callable[[], None], needs: tuple[str, ...] = ()) -> None:
    """Run a check, or a demo that depends on your code, without derailing the notebook.

    A stub you have not filled in yet simply says so. A wrong answer prints the check's own
    message — which names the likely mistake — and the notebook carries on, so one broken
    exercise never hides the feedback on the others. A demo names the exercises it `needs`:
    until each has passed its check, the demo says which one it is waiting for and skips.
    Nothing is swallowed: every outcome is recorded in `_STATUS` for the progress board at the
    foot of the notebook, and every failure in `_FAILED_CHECKS`, which ends a script run
    non-zero.
    """
    waiting = [name for name in _EXERCISES   # in the order you meet them
               if name in needs and _STATUS.get(name) != "passed"]
    if waiting:
        _STATUS[label] = "not started"
        print(f"{label}: skipped — needs {_named(waiting)} to pass first.")
        return
    try:
        check()
    except NotImplementedError as exc:
        _STATUS[label] = "not started"
        stub = traceback.extract_tb(exc.__traceback__)[-1].name   # the frame that raised
        owner = [name for name, funcs in _EXERCISES.items() if stub in funcs and name != label]
        if owner:
            print(f"{label}: skipped — needs {_named(owner)} first.")
        elif label in _EXERCISES:
            print(f"{label}: not implemented yet — fill in {stub}() above, then re-run "
                  "this cell.")
        else:
            print(f"{label}: skipped — {stub}() is not implemented yet.")
    except AssertionError as exc:
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: FAILED — {exc}")
    except Exception as exc:  # a half-finished implementation raising something else
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: raised {type(exc).__name__}: {exc}")
    else:
        _STATUS[label] = "passed"


def _show(fig) -> None:
    """Draw in Jupyter, close silently in a script run. The execution gate opens no windows."""
    fig.tight_layout()
    if _INTERACTIVE:
        plt.show()
    else:
        plt.close(fig)

## 1. The forecast, and the interface it arrives on

Module 5 hands over a *distribution*. Not an RUL, not an RUL with an error bar bolted on
afterwards, but a probability for every hour at which this machine could stop. The interface
is three fields: the unit, the hour the forecast was made, and a pmf over hours ahead.

Module 5's method is an exponential degradation fit whose rate parameter carries posterior
uncertainty; pushing that uncertainty through the crossing time gives a right-skewed
predictive distribution that narrows as the machine degrades. This notebook renders that
shape directly from its mean and standard deviation, because the *shape* is all this module
needs and holding the mean fixed while moving the width is the experiment of section 7.
When module 5 is on disk, `make_forecast` is the only cell that changes: everything below it
consumes `RULForecast` and nothing else.

Read the generator. Two conventions in it decide every number in this lesson:

- `pmf[k]` is the probability of failing **during** hour `k`, so the whole of the tail
  beyond the grid is folded into the last bin. No mass is discarded.
- A machine that fails during hour `t` has **not** made it to an intervention booked for
  hour `t`. Failure at exactly `t` is a failure, not a catch. Every cost function below
  uses that convention and the rubric grades it.

**Optional, and off the graded path.** Nothing in this notebook downloads anything and every
gate runs with the machine offline. Afterwards, if you want the same code pointed at a real
corpus, NASA's Prognostics Center of Excellence publishes the Turbofan Engine Degradation
Simulation (C-MAPSS) sets as a direct download under a disclaimer that users employ the data
at their own risk; `claims.yaml` carries that line and the URL. Fit module 5's degradation
model to one unit's trajectory, summarise the predictive RUL by its mean and standard
deviation, and everything below this cell takes it unchanged.

In [ ]:
def _standard_normal_cdf(x: float) -> float:
    """Phi(x), from math.erf. numpy has no erf and scipy is not on this tier."""
    return 0.5 * (1.0 + math.erf(x / math.sqrt(2.0)))


def make_forecast(unit: int, mean_rul: float, sd_rul: float, made_at: int = NOW,
                  grid_hours: int = GRID_HOURS) -> RULForecast:
    """A predictive RUL distribution with the given mean and standard deviation, on the grid.

    Right-skewed (log-normal), because a degradation rate you are uncertain about turns into a
    crossing time whose right tail is long. Given to you; this is module 5's output port.
    """
    if mean_rul <= 0 or sd_rul <= 0:
        raise ValueError("mean_rul and sd_rul must both be positive")
    sigma_sq = math.log(1.0 + (sd_rul / mean_rul) ** 2)
    sigma = math.sqrt(sigma_sq)
    mu = math.log(mean_rul) - 0.5 * sigma_sq
    cdf = np.empty(grid_hours + 1)
    cdf[0] = 0.0
    for k in range(1, grid_hours + 1):
        cdf[k] = _standard_normal_cdf((math.log(k) - mu) / sigma)
    pmf = np.diff(cdf)
    pmf[-1] += 1.0 - cdf[-1]           # fold the tail beyond the grid into the last bin
    pmf = np.maximum(pmf, 0.0)
    pmf /= pmf.sum()
    return RULForecast(unit=int(unit), made_at=int(made_at),
                       hours=np.arange(grid_hours, dtype=int), pmf=pmf)


class FleetUnit(NamedTuple):
    """One machine: what the model says, and what is actually going to happen to it."""
    forecast: RULForecast
    true_rul: int          # ground truth, unavailable in real life; here so we can score


def generate_fleet(seed: int = SEED, n_units: int = N_FLEET) -> list:
    """Six machines, each with a predictive RUL distribution and a true failure hour.

    The truth is drawn FROM each machine's own forecast, so every forecast here is calibrated.
    That is the generous case, and section 6 shows it is not enough on its own.
    """
    rng = np.random.default_rng(seed)
    fleet = []
    for unit in range(n_units):
        mean_rul = float(rng.integers(170, 340))
        cv = float(rng.uniform(0.12, 0.42))          # coefficient of variation: the width
        forecast = make_forecast(unit, mean_rul, mean_rul * cv)
        true_rul = int(rng.choice(forecast.hours, p=forecast.pmf))
        fleet.append(FleetUnit(forecast=forecast, true_rul=true_rul))
    return fleet


def maintenance_calendar(days: int = CALENDAR_DAYS, open_hour: int = WORKSHOP_OPEN_HOUR,
                         working_days: Sequence = WORKING_DAYS) -> np.ndarray:
    """The absolute hours at which the workshop can take a machine. Weekdays only.

    Hour 0 is a Monday midnight, so day `d` is a working day when `d % 7` is in
    `working_days`. Given to you.
    """
    return np.array([d * 24 + open_hour for d in range(days) if d % 7 in working_days],
                    dtype=int)


FLEET = generate_fleet()
CALENDAR = maintenance_calendar()
SLOTS = CALENDAR[:SHORTLIST]

print(f"fleet          {len(FLEET)} machines, one forecast each, all made at hour {NOW}")
print(f"forecast grid  {FLEET[0].forecast.hours.size} hourly bins, "
      f"pmf sums to {FLEET[0].forecast.pmf.sum():.6f}")
print(f"calendar       {CALENDAR.size} slots in {CALENDAR_DAYS} days "
      f"({len(WORKING_DAYS)} working days a week at {WORKSHOP_OPEN_HOUR:02d}:00)")
print(f"shortlist      {SLOTS.size} slots the planner will negotiate over: {[int(s) for s in SLOTS]}")
print(f"prices         planned {SCENARIO.planned:,.0f} · unplanned {SCENARIO.unplanned:,.0f} "
      f"· {SCENARIO.life_per_hour:,.0f} per hour of life thrown away")

## 2. Exercise 1 — `forecast_summary()`

Start with the numbers a dashboard shows, because they are the ones you are about to stop
believing. The mean RUL, the spread, and three quantiles.

The quantile rule is the exercise. `p10` is the **smallest** hour whose cumulative
probability has reached 0.10 — reached, not passed. Off by one here and every interval you
quote is narrower than it should be on one side and wider on the other, which is the sort of
bug that survives for years because nobody can see it.

<details><summary>💡 Hint 1 — what to think about</summary>

A pmf is already a set of weights, so does the mean need dividing by anything afterwards?
For the percentiles, the rule is the smallest hour whose cumulative probability has
REACHED q, and a running sum of floats can land a hair under a boundary that ought to be
exact. Finally: what should happen to a pmf that does not sum to one, or that has a
negative bin, and whose bug is it?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Validate before computing: equal lengths, an explicit check that no bin is negative (a
negative bin can hide inside a pmf that still sums to one), and a total within the
docstring's tolerance, each failure a `ValueError` rather than a quiet renormalise. Take
the pmf-weighted mean, then the square root of the pmf-weighted squared deviations. For
each percentile, find the first position where the cumulative sum reaches q less a tiny
slack, and report the hour stored at that position as an int, not the position.
</details>

In [ ]:
def forecast_summary(forecast: RULForecast) -> Summary:
    """Mean, standard deviation and the 10th, 50th and 90th percentiles of an RUL forecast.

    The mean is `sum(hours * pmf)`; the standard deviation is the square root of
    `sum(pmf * (hours - mean)**2)`. A quantile `q` is the SMALLEST `hours[k]` whose cumulative
    probability is at least `q` — use `>=`, allowing a little floating-point slack.

    Raise `ValueError` if `hours` and `pmf` differ in length, if any probability is negative,
    or if the pmf does not sum to 1 within 1e-6. A forecast that does not sum to 1 is a bug
    upstream, and silently renormalising it here hides that bug from module 5.

    Returns a `Summary(mean, sd, p10, p50, p90)` with `mean` and `sd` floats and the three
    percentiles ints taken from `hours`.

    Example:
        >>> f = RULForecast(0, 0, np.array([0, 1, 2, 3]), np.array([0.1, 0.4, 0.4, 0.1]))
        >>> s = forecast_summary(f)
        >>> round(s.mean, 3), (s.p10, s.p50, s.p90)
        (1.5, (0, 1, 2))
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_forecast_summary() -> None:
    toy = RULForecast(0, 0, np.array([0, 1, 2, 3]), np.array([0.1, 0.4, 0.4, 0.1]))
    got = forecast_summary(toy)
    assert abs(got.mean - 1.5) < 1e-9, f"mean of the toy pmf is 1.5, got {got.mean!r}"
    assert abs(got.sd - math.sqrt(0.65)) < 1e-9, (
        f"sd is sqrt(sum(pmf*(hours-mean)**2)) = {math.sqrt(0.65):.4f}, got {got.sd!r}. A "
        "value near 0.806 squared means you divided by n somewhere: a pmf is already weighted"
    )
    assert (got.p10, got.p50, got.p90) == (0, 1, 2), (
        f"quantiles came back {(got.p10, got.p50, got.p90)}; expected (0, 1, 2). The rule is "
        "the smallest hour whose cumulative probability has REACHED q: the cdf is "
        "[0.1, 0.5, 0.9, 1.0], so p10 is hour 0 and p50 is hour 1, not hour 2"
    )
    one = RULForecast(0, 0, np.array([7]), np.array([1.0]))
    assert forecast_summary(one).sd == 0.0, "a forecast with all its mass on one hour has sd 0"
    for bad, why in (
            (RULForecast(0, 0, np.array([0, 1]), np.array([0.5, 0.4])), "sums to 0.9"),
            (RULForecast(0, 0, np.array([0, 1]), np.array([1.4, -0.4])), "has a negative bin"),
            (RULForecast(0, 0, np.array([0, 1, 2]), np.array([0.6, -0.2, 0.6])),
             "has a negative bin but still sums to 1 and still has a positive variance, so "
             "only an explicit sign check catches it"),
            (RULForecast(0, 0, np.array([0, 1, 2]), np.array([0.5, 0.5])), "is the wrong length")):
        try:
            forecast_summary(bad)
        except ValueError:
            continue
        raise AssertionError(f"a pmf that {why} must raise ValueError, not be repaired here")
    slack = forecast_summary(RULForecast(0, 0, np.arange(5),
                                         np.array([1, 1, 3, 1, 6], dtype=float) / 12.0))
    assert slack.p50 == 3, (
        f"the median of [1, 1, 3, 1, 6]/12 is hour 3 — the first four bins are exactly half "
        f"the mass — but np.cumsum puts that boundary at 0.5 - 5.6e-17. Got {slack.p50}. "
        "Compare the cdf against `q - 1e-12` rather than against q"
    )
    real = forecast_summary(FLEET[0].forecast)
    assert real.p10 < real.p50 < real.p90, (
        f"on a real forecast the percentiles must be ordered; got {real.p10}, {real.p50}, "
        f"{real.p90}"
    )
    print(f"exercise 1 looks right — unit 0: mean {real.mean:.1f} h, sd {real.sd:.1f} h, "
          f"P10 {real.p10} h, P50 {real.p50} h, P90 {real.p90} h")

In [ ]:
_try("exercise 1", _check_forecast_summary)

## 3. Exercise 2 — `survival_and_hazard()`

Two curves derived from the same pmf, and the second one is where the answer to this whole
module lives.

**Survival** `S[k] = P(RUL > hours[k])`: the probability the machine is still running after
hour `k`. It starts just under 1 and ends at 0.

**Hazard** `h[k] = P(fails during hour k | still running at the start of hour k)`. The
denominator is the population still at risk at the *start* of the hour, which is
`S[k] + pmf[k]`, not `S[k]`. Use `S[k]` and your hazard is shifted by one bin and diverges
at the tail. Where the at-risk population is zero the hazard is 0, not a nan.

<details><summary>💡 Hint 1 — what to think about</summary>

Survival at hour k is the chance of lasting BEYOND hour k, so it already excludes the
machines that fail during hour k. The hazard asks about the machines still running at the
START of hour k: which population is that, in terms of the two arrays you have? And what
happens to your division at the tail, once nothing is left running?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Survival is one minus the running total of the pmf, clipped into the unit interval so a
floating-point overshoot cannot come back as a tiny negative probability. The at-risk
population for hour k is that hour's survival plus that hour's own probability. Divide the
pmf by it only where it is positive and leave a zero everywhere else, rather than dividing
everywhere and repairing the nans afterwards. Return both as float arrays the length of
the pmf.
</details>

In [ ]:
def survival_and_hazard(forecast: RULForecast) -> tuple:
    """Survival and discrete hazard of an RUL forecast, as two float arrays.

    `survival[k] = P(RUL > hours[k]) = 1 - cumsum(pmf)[k]`, clipped into [0, 1] so that
    floating-point crumbs at the tail do not come back as -1e-17.

    `hazard[k] = pmf[k] / (survival[k] + pmf[k])` — the chance of failing during hour k given
    that the machine reached the start of hour k. Where the denominator is 0, the hazard is
    0.0 rather than nan: no machines left means no risk, not an undefined one.

    Returns `(survival, hazard)`, both the same length as `pmf`.

    Example:
        >>> f = RULForecast(0, 0, np.array([0, 1, 2]), np.array([0.2, 0.3, 0.5]))
        >>> s, h = survival_and_hazard(f)
        >>> [round(v, 4) for v in s], [round(v, 4) for v in h]
        ([0.8, 0.5, 0.0], [0.2, 0.375, 1.0])
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_survival_and_hazard() -> None:
    toy = RULForecast(0, 0, np.array([0, 1, 2]), np.array([0.2, 0.3, 0.5]))
    surv, haz = survival_and_hazard(toy)
    assert np.allclose(surv, [0.8, 0.5, 0.0]), (
        f"survival came back {np.round(surv, 4).tolist()}; expected [0.8, 0.5, 0.0]. "
        "S[k] = 1 - cumsum(pmf)[k] — strictly GREATER than hours[k], so the last entry is 0"
    )
    assert np.allclose(haz, [0.2, 0.375, 1.0]), (
        f"hazard came back {np.round(haz, 4).tolist()}; expected [0.2, 0.375, 1.0]. "
        "[0.25, 0.6, inf] means you divided by survival[k] instead of by the population still "
        "at risk at the START of hour k, which is survival[k] + pmf[k]"
    )
    assert np.all(np.isfinite(haz)), (
        "the last hazard bin divides 0.5 by 0.5, not by 0 — if you got inf or nan, the "
        "denominator is survival[k] rather than survival[k] + pmf[k]"
    )
    tail = RULForecast(0, 0, np.array([0, 1, 2]), np.array([1.0, 0.0, 0.0]))
    _, haz_tail = survival_and_hazard(tail)
    assert np.all(np.isfinite(haz_tail)) and haz_tail[1] == 0.0 and haz_tail[2] == 0.0, (
        f"after all the mass is gone the hazard is 0.0, not nan; got "
        f"{np.round(haz_tail, 4).tolist()}"
    )
    ninths, _ = survival_and_hazard(RULForecast(0, 0, np.arange(9), np.full(9, 1.0 / 9.0)))
    assert float(np.asarray(ninths).min()) >= 0.0, (
        f"nine equal bins: np.cumsum overshoots 1 by 2.2e-16, so a bare 1 - cumsum(pmf) ends "
        f"at {float(np.asarray(ninths).min()):.3e}, and a probability is never negative. "
        "Clip the result into [0, 1]"
    )
    real_s, real_h = survival_and_hazard(FLEET[0].forecast)
    assert real_s[0] <= 1.0 and real_s[-1] >= 0.0 and np.all(np.diff(real_s) <= 1e-12), (
        "survival must be non-increasing and inside [0, 1]"
    )
    p50 = forecast_summary(FLEET[0].forecast).p50
    assert abs(real_s[p50] - 0.5) < 0.02, (
        f"survival at the median hour should be about 0.5; got {real_s[p50]:.4f} at hour "
        f"{p50}. A gap this size usually means survival is off by one bin"
    )
    print(f"exercise 2 looks right — unit 0 survival at hour {p50} is {real_s[p50]:.3f}, "
          f"hazard there is {real_h[p50] * 1000:.3f} per 1000 running hours")

In [ ]:
_try("exercise 2", _check_survival_and_hazard)

## 4. Exercise 3 — `expected_cost_at()`

Now the money. You commit today to pulling this machine in `t` hours. Two things can happen.

- **It fails first** (RUL `<= t`): you pay `unplanned`. It does not matter by how much you
  missed; the machine is down in service either way.
- **You get there first** (RUL `> t`): you pay `planned`, plus `life_per_hour` for every
  hour of remaining useful life still on the machine when you pulled it — `RUL - t` of them.

Average those over the distribution and you have the expected cost of the plan. That third
price is what replaces module 1's false alarm: against a distribution, *every* intervention
is early by some amount, and the question is only how much you are willing to waste.

<details><summary>💡 Hint 1 — what to think about</summary>

Split the distribution at the intervention hour. Which side does a failure DURING the
booked hour fall on: did you collect that machine or not? On the side you do catch, what
are you paying for besides the planned job, and who pays it: the machines you caught, or
the ones that had already failed? Hour 0 is a real hour with real probability in it.
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Reject a negative `hours_ahead` with `ValueError`. Mark the hours at or before the
intervention: that mass pays the unplanned price. Every other hour pays the planned price
plus the life-per-hour charge on the hours of life still left on the machine. Weight both
by the pmf, sum, and return a plain float. Past the end of the grid every hour is marked,
so the answer falls out without a special case.
</details>

In [ ]:
def expected_cost_at(forecast: RULForecast, hours_ahead: int, prices: Prices) -> float:
    """Expected cost of committing now to intervene `hours_ahead` hours from `made_at`.

    Sum over the pmf of:
      * `prices.unplanned`                                   where `hours <= hours_ahead`
      * `prices.planned + prices.life_per_hour * (hours - hours_ahead)`   where `hours > ...`

    Failing at exactly `hours_ahead` counts as a failure: the machine went down during the
    hour you were going to collect it. `hours_ahead` beyond the end of the grid therefore
    costs exactly `prices.unplanned`.

    Raise `ValueError` for a negative `hours_ahead`.

    Returns a plain float.

    Example:
        >>> f = RULForecast(0, 0, np.array([0, 1, 2, 3]), np.array([0.1, 0.4, 0.4, 0.1]))
        >>> p = Prices(planned=10.0, unplanned=100.0, life_per_hour=1.0)
        >>> round(expected_cost_at(f, 1, p), 4)   # 0.5*100 + 0.5*10 + (0.4*1 + 0.1*2)
        55.6
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_expected_cost_at() -> None:
    toy = RULForecast(0, 0, np.array([0, 1, 2, 3]), np.array([0.1, 0.4, 0.4, 0.1]))
    cheap = Prices(planned=10.0, unplanned=100.0, life_per_hour=1.0)
    got = expected_cost_at(toy, 1, cheap)
    assert abs(got - 55.6) < 1e-9, (
        f"expected 55.6, got {got!r}. 0.5 of the mass is at hours 0 and 1 and pays 100; the "
        "other 0.5 pays 10 plus 1 per hour of life left, which is 0.4*1 + 0.1*2"
    )
    at_zero = expected_cost_at(toy, 0, cheap)
    assert abs(at_zero - (0.1 * 100 + 0.9 * 10 + (0.4 * 1 + 0.4 * 2 + 0.1 * 3))) < 1e-9, (
        f"intervening at hour 0 still loses to the 0.1 of mass already in hour 0; got "
        f"{at_zero!r}. Hour 0 is a real hour and `hours <= 0` is not empty"
    )
    boundary = expected_cost_at(toy, 2, cheap)
    assert abs(boundary - (0.9 * 100 + 0.1 * 10 + 0.1 * 1)) < 1e-9, (
        f"expected 91.1 at hours_ahead=2, got {boundary!r}. A value of 55.6 or 51.6 means "
        "failure AT hours_ahead is being counted as a catch — the machine went down during "
        "the hour you were going to collect it, so the comparison is `hours <= hours_ahead`"
    )
    far = expected_cost_at(toy, 99, cheap)
    assert abs(far - 100.0) < 1e-9, (
        f"past the end of the grid every unit has already failed, so the cost is exactly the "
        f"unplanned price; got {far!r}"
    )
    try:
        expected_cost_at(toy, -1, cheap)
    except ValueError:
        pass
    else:
        raise AssertionError("a negative hours_ahead must raise ValueError, not be clamped")
    real = FLEET[0].forecast
    now_cost = expected_cost_at(real, 0, SCENARIO)
    never_cost = expected_cost_at(real, GRID_HOURS, SCENARIO)
    mid = expected_cost_at(real, 150, SCENARIO)
    assert mid < now_cost and mid < never_cost, (
        f"on unit 0 the cost curve must dip in the middle: pulling it now costs "
        f"{now_cost:,.0f}, never pulling it costs {never_cost:,.0f}, and hour 150 costs "
        f"{mid:,.0f}. If the middle is not cheapest, the wasted-life term is missing or is "
        "being charged to the units that already failed"
    )
    print(f"exercise 3 looks right — unit 0: pull now {now_cost:,.0f}, "
          f"pull at 150 h {mid:,.0f}, never pull {never_cost:,.0f}")

In [ ]:
_try("exercise 3", _check_expected_cost_at)

## 5. Exercise 4 — `best_intervention_time()`

Sweep the candidate hours, keep the cheapest. The convention is module 1's: as you sweep,
keep a point only if it is **strictly** cheaper than the best so far, so a tie goes to the
earlier hour. Earlier is the safe side of a tie, and picking one deliberately beats letting
`argmin` pick for you.

<details><summary>💡 Hint 1 — what to think about</summary>

This is module 1's sweep and module 1's tie rule. When two candidate hours cost exactly
the same, which one is the safe side to take, and does your comparison keep the first of a
tie or the last? Would sorting the candidates, or searching them backwards, quietly change
the answer? And what should an empty sweep produce?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Raise `ValueError` on an empty candidate list. Walk the candidates in the order given,
pricing each with your `expected_cost_at`, and replace the running best only when a
candidate is strictly cheaper. Return a `Plan` holding that hour as an int together with
its own cost as a float.
</details>

In [ ]:
def best_intervention_time(forecast: RULForecast, candidates, prices: Prices) -> Plan:
    """The cheapest intervention time among `candidates`, in hours ahead of `made_at`.

    Sweep `candidates` in the order given and keep a candidate only when it is STRICTLY
    cheaper than the best so far, so a tie is won by whichever came first. Raise `ValueError`
    on an empty `candidates`.

    Returns a `Plan(hours_ahead, cost)` with `hours_ahead` an int and `cost` a float.

    Example:
        >>> f = RULForecast(0, 0, np.array([0, 1, 2, 3]), np.array([0.1, 0.4, 0.4, 0.1]))
        >>> p = Prices(planned=10.0, unplanned=100.0, life_per_hour=1.0)
        >>> best_intervention_time(f, [0, 1, 2, 3], p)
        Plan(hours_ahead=0, cost=20.5)
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_best_intervention_time() -> None:
    toy = RULForecast(0, 0, np.array([0, 1, 2, 3]), np.array([0.1, 0.4, 0.4, 0.1]))
    cheap = Prices(planned=10.0, unplanned=100.0, life_per_hour=1.0)
    plan = best_intervention_time(toy, [0, 1, 2, 3], cheap)
    assert (plan.hours_ahead, round(plan.cost, 6)) == (0, 20.5), (
        f"expected Plan(0, 20.5) on the toy forecast, got {plan}. With a cheap unplanned "
        "price the sweep should pull early"
    )
    flat = RULForecast(0, 0, np.array([0, 1, 2]), np.array([0.0, 0.0, 1.0]))
    free = Prices(planned=10.0, unplanned=10.0, life_per_hour=0.0)
    tie = best_intervention_time(flat, [0, 1, 2], free)
    assert tie.hours_ahead == 0, (
        f"every candidate costs the same here, so the tie goes to the EARLIEST — expected 0, "
        f"got {tie.hours_ahead}. `<=` in the sweep, or np.argmin over a reversed list, gives "
        "the late answer and the late answer is the one that fails in service"
    )
    try:
        best_intervention_time(toy, [], cheap)
    except ValueError:
        pass
    else:
        raise AssertionError("an empty candidate list must raise ValueError")
    real = FLEET[0].forecast
    plan = best_intervention_time(real, CANDIDATES, SCENARIO)
    summary = forecast_summary(real)
    sweep = np.array([expected_cost_at(real, int(t), SCENARIO) for t in CANDIDATES])
    assert abs(plan.cost - sweep.min()) < 1e-9, (
        f"your plan costs {plan.cost:,.2f} but the cheapest point on the same sweep is "
        f"{sweep.min():,.2f}"
    )
    assert 0 < plan.hours_ahead < int(summary.mean), (
        f"unit 0's optimal intervention time came back {plan.hours_ahead} h against a mean "
        f"RUL of {summary.mean:.0f} h. On this scenario the optimum is strictly inside the "
        "interval — not hour 0, and well before the mean"
    )
    print(f"exercise 4 looks right — unit 0: intervene at {plan.hours_ahead} h for "
          f"{plan.cost:,.0f}, against a mean RUL of {summary.mean:.0f} h")

In [ ]:
_try("exercise 4", _check_best_intervention_time)

## 6. The answer is not the mean, and here is why

Two routes to the same number. The sweep you just wrote is one. The other is a one-line
piece of calculus on the cost function: moving the plan one hour later buys you one more
hour of life on every machine that survives, and costs you `unplanned - planned` on the
share that fails in that hour. Those balance when the **hazard rate** reaches

    life_per_hour / (unplanned - planned)

and that is a property of the *shape* of the distribution, not of its mean. The cell below
computes the crossing from your `survival_and_hazard` and compares it with your sweep.

In [ ]:
# The cells that plan one machine at a time use YOUR best_intervention_time, which prices
# with YOUR expected_cost_at, and report YOUR forecast_summary beside it.
_FOR_PLAN = ("exercise 1", "exercise 3", "exercise 4")


def hazard_crossing(forecast: RULForecast, prices: Prices) -> int:
    """First hour at which the hazard reaches life_per_hour / (unplanned - planned).

    Given to you, on top of YOUR survival_and_hazard, so the two routes are independent.
    """
    _, hazard = survival_and_hazard(forecast)
    target = prices.life_per_hour / (prices.unplanned - prices.planned)
    over = np.flatnonzero(hazard >= target)
    return int(forecast.hours[over[0]]) if over.size else int(forecast.hours[-1])


def _two_routes_agree() -> None:
    target = SCENARIO.life_per_hour / (SCENARIO.unplanned - SCENARIO.planned)
    print(f"balance hazard = {SCENARIO.life_per_hour:,.0f} / "
          f"({SCENARIO.unplanned:,.0f} - {SCENARIO.planned:,.0f}) = {target:.6f} per hour")
    print(f"{'unit':>4} {'mean RUL':>9} {'sd':>7} {'sweep t*':>9} {'hazard t*':>10} "
          f"{'cost(t*)':>10} {'cost(mean)':>11} {'ratio':>6}")
    worst = 0.0
    for fu in FLEET:
        summary = forecast_summary(fu.forecast)
        plan = best_intervention_time(fu.forecast, CANDIDATES, SCENARIO)
        cross = hazard_crossing(fu.forecast, SCENARIO)
        at_mean = expected_cost_at(fu.forecast, int(round(summary.mean)), SCENARIO)
        worst = max(worst, at_mean / plan.cost)
        print(f"{fu.forecast.unit:>4} {summary.mean:>9.1f} {summary.sd:>7.1f} "
              f"{plan.hours_ahead:>9d} {cross:>10d} {plan.cost:>10,.0f} {at_mean:>11,.0f} "
              f"{at_mean / plan.cost:>6.2f}")
        assert abs(cross - plan.hours_ahead) <= 2, (
            f"unit {fu.forecast.unit}: the sweep says {plan.hours_ahead} h and the hazard "
            f"crossing says {cross} h. These are two routes to one number and they must agree "
            "within a bin or two; a large gap means the hazard denominator or the cost "
            "convention is wrong"
        )
    print(f"\nnot one machine is best served at its mean RUL. Planning at the mean costs up "
          f"to {worst:.2f}x\nthe optimum on this fleet — and the mean is the number the "
          f"dashboard shows.")


_try("the mean is not the plan", _two_routes_agree, needs=_FOR_PLAN + ("exercise 2",))

In [ ]:
def _plot_cost_curve() -> None:
    fu = FLEET[0]
    summary = forecast_summary(fu.forecast)
    plan = best_intervention_time(fu.forecast, CANDIDATES, SCENARIO)
    costs = np.array([expected_cost_at(fu.forecast, int(t), SCENARIO) for t in CANDIDATES])
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(7.2, 6.0), sharex=True)
    ax1.plot(fu.forecast.hours[:601], fu.forecast.pmf[:601], color="steelblue")
    ax1.axvline(summary.mean, color="darkorange", ls="--", label=f"mean RUL {summary.mean:.0f} h")
    ax1.axvline(plan.hours_ahead, color="crimson", label=f"optimal pull {plan.hours_ahead} h")
    ax1.set_ylabel("P(fails this hour)")
    ax1.set_title(f"unit {fu.forecast.unit}: the forecast, and what to do about it")
    ax1.legend(fontsize=8)
    ax2.plot(CANDIDATES, costs / 1e3, color="black")
    ax2.axvline(summary.mean, color="darkorange", ls="--")
    ax2.axvline(plan.hours_ahead, color="crimson")
    ax2.set_xlabel("hours ahead of now")
    ax2.set_ylabel("expected cost, thousands")
    _show(fig)
    print(f"plotted: the cost curve dips at {plan.hours_ahead} h and is "
          f"{expected_cost_at(fu.forecast, int(round(summary.mean)), SCENARIO) / plan.cost:.2f}x "
          f"higher at the mean")


_try("cost curve", _plot_cost_curve, needs=_FOR_PLAN)

## 7. Width, not just centre

Hold the mean at 300 hours and move only the spread. If the intervention time were a
property of the centre, every row of the next table would be the same. Then hold the spread
and move the centre, so you can see that both matter and neither substitutes for the other.

In [ ]:
def _width_and_centre() -> None:
    print("mean held at 300 h; only the width moves")
    print(f"{'sd':>5} {'P10':>6} {'P90':>6} {'t*':>6} {'t*/mean':>9} {'cost(t*)':>10}")
    times = []
    for sd in (30, 60, 90, 120, 150):
        f = make_forecast(99, 300.0, float(sd))
        s = forecast_summary(f)
        plan = best_intervention_time(f, CANDIDATES, SCENARIO)
        times.append(plan.hours_ahead)
        print(f"{sd:>5} {s.p10:>6} {s.p90:>6} {plan.hours_ahead:>6} "
              f"{plan.hours_ahead / s.mean:>9.2f} {plan.cost:>10,.0f}")
    assert times == sorted(times, reverse=True) and times[0] > times[-1], (
        f"the optimal intervention times came back {times}. Holding the mean fixed and "
        "widening the distribution must pull the intervention EARLIER every time: a wider "
        "forecast puts mass in the near tail, which raises the hazard sooner"
    )
    print(f"\nsame mean, same prices, and {times[0] - times[-1]} hours between the tightest "
          f"forecast's plan and the widest.\nThat gap is the price of uncertainty, and it is "
          f"invisible to anyone reading the mean.\n")
    print("sd held at 90 h; only the centre moves")
    print(f"{'mean':>6} {'t*':>6} {'t*/mean':>9} {'cost(t*)':>10}")
    centres = []
    for mean in (200, 300, 400):
        f = make_forecast(98, float(mean), 90.0)
        plan = best_intervention_time(f, CANDIDATES, SCENARIO)
        centres.append(plan.hours_ahead)
        print(f"{mean:>6} {plan.hours_ahead:>6} {plan.hours_ahead / mean:>9.2f} "
              f"{plan.cost:>10,.0f}")
    assert centres == sorted(centres) and centres[0] < centres[-1], (
        f"moving the centre later must move the plan later; got {centres}"
    )
    print("\nthe centre matters too — but notice t*/mean is not constant, so you cannot "
          "recover\nthe plan from the mean with a fixed safety factor either.")


_try("width and centre", _width_and_centre, needs=_FOR_PLAN)

In [ ]:
def _plot_width() -> None:
    fig, ax = plt.subplots(figsize=(7.2, 3.6))
    for sd, colour in ((30, "seagreen"), (90, "steelblue"), (150, "crimson")):
        f = make_forecast(99, 300.0, float(sd))
        costs = np.array([expected_cost_at(f, int(t), SCENARIO) for t in CANDIDATES])
        plan = best_intervention_time(f, CANDIDATES, SCENARIO)
        ax.plot(CANDIDATES, costs / 1e3, color=colour, label=f"sd {sd} h, pull at {plan.hours_ahead} h")
        ax.plot([plan.hours_ahead], [plan.cost / 1e3], "o", color=colour)
    ax.axvline(300, color="darkorange", ls="--", label="mean RUL 300 h (all three)")
    ax.set_xlabel("hours ahead of now")
    ax.set_ylabel("expected cost, thousands")
    ax.set_title("one mean, three widths, three different answers")
    ax.legend(fontsize=8)
    _show(fig)
    print("plotted: three forecasts with the same mean and three different optima")


_try("width plot", _plot_width, needs=("exercise 3", "exercise 4"))

## 8. Exercise 5 — `realised_cost()`, and what a confident wrong answer costs

An expected cost is a number about a belief. Once the machine has actually stopped there is
a second number, about the world: what the plan cost *given what happened*. You need both,
because they can disagree spectacularly, and the disagreement is the whole argument for
carrying the width around instead of the mean.

Same convention as before. Fail at or before the intervention hour and you pay `unplanned`;
otherwise you pay `planned` plus the life you threw away.

<details><summary>💡 Hint 1 — what to think about</summary>

This is exercise 3 for one known outcome, and averaging it over a forecast must give
exercise 3's answer back exactly. So the same boundary rule has to apply: a machine that
stops during the hour you booked, collected or failed? Two functions that disagree about
that single hour end up a whole bin's worth of probability apart.
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Reject a negative `hours_ahead` or a negative `true_rul` with `ValueError`. If the
machine's true remaining life ends at or before the booked hour, it is the unplanned
price. Otherwise it is the planned price plus the life-per-hour charge on the hours you
threw away. Use exactly the comparison your `expected_cost_at` uses.
</details>

In [ ]:
def realised_cost(hours_ahead: int, true_rul: int, prices: Prices) -> float:
    """What a plan actually cost, once you know when the machine really stopped.

    `prices.unplanned` when `true_rul <= hours_ahead` — the machine went down during or
    before the hour you were going to collect it. Otherwise `prices.planned` plus
    `prices.life_per_hour * (true_rul - hours_ahead)`.

    This is the per-outcome version of `expected_cost_at`, and averaging it over a forecast's
    pmf must reproduce `expected_cost_at` exactly. Raise `ValueError` on a negative
    `hours_ahead` or a negative `true_rul`.

    Returns a plain float.

    Example:
        >>> p = Prices(planned=10.0, unplanned=100.0, life_per_hour=1.0)
        >>> realised_cost(5, 12, p), realised_cost(5, 5, p), realised_cost(5, 2, p)
        (17.0, 100.0, 100.0)
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_realised_cost() -> None:
    cheap = Prices(planned=10.0, unplanned=100.0, life_per_hour=1.0)
    assert realised_cost(5, 12, cheap) == 17.0, (
        f"a machine with 12 h left pulled at 5 h costs planned + 7 hours of life = 17.0, got "
        f"{realised_cost(5, 12, cheap)!r}"
    )
    assert realised_cost(5, 5, cheap) == 100.0, (
        f"a machine that fails at exactly the hour you booked has failed; expected 100.0, got "
        f"{realised_cost(5, 5, cheap)!r}. This must match expected_cost_at's `<=` convention "
        "or the two functions will disagree by one bin's worth of probability"
    )
    assert realised_cost(5, 2, cheap) == 100.0, "a machine that failed at hour 2 is long gone"
    assert realised_cost(0, 1, cheap) == 11.0, (
        "pulling at hour 0 a machine with 1 h left costs planned plus one hour of life"
    )
    for bad in ((-1, 5), (5, -1)):
        try:
            realised_cost(*bad, cheap)
        except ValueError:
            continue
        raise AssertionError(f"realised_cost{bad + (cheap,)} must raise ValueError")
    # The identity that ties the two cost functions together: average the realised cost over
    # the forecast and you must get the expected cost back, to the last bit.
    f = FLEET[0].forecast
    for t in (0, 77, 200, 599):
        by_hand = float(sum(float(p) * realised_cost(t, int(h), SCENARIO)
                            for h, p in zip(f.hours, f.pmf) if p > 0))
        direct = expected_cost_at(f, t, SCENARIO)
        assert abs(by_hand - direct) < 1e-6, (
            f"at t={t} the pmf-weighted realised cost is {by_hand:,.4f} but expected_cost_at "
            f"says {direct:,.4f}. The two functions must use the SAME rule about failing at "
            "exactly t; if one uses `<` and the other `<=`, they differ by pmf[t] times the "
            "gap between the two prices"
        )
    print("exercise 5 looks right — the realised cost averages back to the expected cost at "
          "every hour tried")

In [ ]:
_try("exercise 5", _check_realised_cost)

In [ ]:
# Three forecasts of the SAME machine, which is in fact 190 hours from failure. One is tight
# and wrong, one is tight and right, one is loose and roughly right. Price all three plans.
TRUE_RUL_DEMO = 190
DEMO_FORECASTS = (("confident, wrong", 320.0, 25.0),
                  ("confident, right", 190.0, 25.0),
                  ("uncertain, roughly right", 260.0, 110.0))


def _confidence_is_not_accuracy() -> None:
    print(f"the machine will actually stop in {TRUE_RUL_DEMO} h. Three forecasts of it:\n")
    print(f"{'forecast':>26} {'mean':>6} {'sd':>5} {'P10':>5} {'P90':>5} {'t*':>5} "
          f"{'expected':>10} {'realised':>10}")
    results = {}
    for name, mean, sd in DEMO_FORECASTS:
        f = make_forecast(90, mean, sd)
        s = forecast_summary(f)
        plan = best_intervention_time(f, CANDIDATES, SCENARIO)
        got = realised_cost(plan.hours_ahead, TRUE_RUL_DEMO, SCENARIO)
        results[name] = (plan, got)
        print(f"{name:>26} {s.mean:>6.0f} {s.sd:>5.0f} {s.p10:>5} {s.p90:>5} "
              f"{plan.hours_ahead:>5} {plan.cost:>10,.0f} {got:>10,.0f}")
    wrong = results["confident, wrong"]
    loose = results["uncertain, roughly right"]
    assert wrong[1] == SCENARIO.unplanned, (
        f"the confident wrong forecast should book its intervention AFTER the machine has "
        f"already failed, so it pays the full unplanned price; got {wrong[1]:,.0f}"
    )
    assert loose[0].cost > wrong[0].cost and loose[1] < wrong[1], (
        "the uncertain forecast must look WORSE on expected cost and turn out BETTER on "
        f"realised cost; got expected {loose[0].cost:,.0f} vs {wrong[0].cost:,.0f} and "
        f"realised {loose[1]:,.0f} vs {wrong[1]:,.0f}"
    )
    print(f"\nthe confident wrong forecast quotes the cheapest plan on the page "
          f"({wrong[0].cost:,.0f}) and costs\n{wrong[1]:,.0f} — "
          f"{wrong[1] / loose[1]:.1f}x what the vague one cost. Narrow error bars are a claim "
          f"about\nthe world, and pricing a plan takes them at their word.")


_try("confidence is not accuracy", _confidence_is_not_accuracy, needs=_FOR_PLAN + ("exercise 5",))

## 9. Exercise 6 — `slot_costs()`: the calendar is not a clock

Everything so far assumed you can pull a machine at any hour you like. You cannot. The
workshop opens at 08:00 on weekdays, and the hours between Friday morning and Monday morning
do not exist as far as planning is concerned. Price each real slot instead.

A slot before the forecast was made is not a choice, and it is not cost zero either. Mark it
`np.inf`, which is the only value that will survive every `min` downstream without quietly
being chosen.

<details><summary>💡 Hint 1 — what to think about</summary>

Slots are absolute hours on the workshop's clock; the forecast counts hours from the
moment it was made. What has to happen to a slot before it can be priced? And what should
a slot that is already in the past cost, given that it must never win a `min` further
down? Zero would win it, and so would pretending the slot is now.
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

For each slot, work out how many hours after `made_at` it falls. A slot before `made_at`
is unreachable: record infinity and move on, without clamping it to hour 0. Price every
other slot with your `expected_cost_at`. Return a float array with one entry per slot, in
the order the slots were given.
</details>

In [ ]:
def slot_costs(forecast: RULForecast, slots, prices: Prices) -> np.ndarray:
    """Expected cost of taking this machine in each calendar slot.

    `slots` are ABSOLUTE hours; the forecast is indexed in hours ahead of `forecast.made_at`,
    so slot `s` is `s - forecast.made_at` hours ahead. A slot strictly before `made_at` gets
    `np.inf`: you cannot book a machine into last Tuesday.

    Returns a float array the same length as `slots`.

    Example:
        >>> f = RULForecast(0, 10, np.array([0, 1, 2, 3]), np.array([0.1, 0.4, 0.4, 0.1]))
        >>> p = Prices(planned=10.0, unplanned=100.0, life_per_hour=1.0)
        >>> slot_costs(f, [8, 10, 11], p).round(2)
        array([  inf, 20.5 , 55.6 ])
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_slot_costs() -> None:
    f = RULForecast(0, 10, np.array([0, 1, 2, 3]), np.array([0.1, 0.4, 0.4, 0.1]))
    cheap = Prices(planned=10.0, unplanned=100.0, life_per_hour=1.0)
    got = slot_costs(f, [8, 10, 11], cheap)
    assert got.shape == (3,), f"one cost per slot, got shape {got.shape}"
    assert math.isinf(got[0]), (
        f"slot 8 is two hours before this forecast was made (made_at=10), so it is not a "
        f"choice: expected inf, got {got[0]!r}. Zero, or a negative hours_ahead, would make "
        "the past look like the cheapest option available"
    )
    assert abs(got[1] - 20.5) < 1e-9 and abs(got[2] - 55.6) < 1e-9, (
        f"slots 10 and 11 are 0 and 1 hours ahead of made_at=10, costing 20.5 and 55.6; got "
        f"{np.round(got, 3).tolist()}. Using the slot's absolute hour as hours_ahead is the "
        "usual cause — the forecast's clock starts at made_at, not at midnight"
    )
    real = slot_costs(FLEET[0].forecast, SLOTS, SCENARIO)
    assert np.all(np.isfinite(real)), "every shortlisted slot is in the future of hour 0"
    plan = best_intervention_time(FLEET[0].forecast, CANDIDATES, SCENARIO)
    assert real.min() >= plan.cost - 1e-9, (
        f"the best calendar slot ({real.min():,.0f}) cannot beat the best hour on a free "
        f"clock ({plan.cost:,.0f}) — the slots are a subset of the hours"
    )
    best_slot = int(SLOTS[int(np.argmin(real))])
    print(f"exercise 6 looks right — unit 0's best slot is hour {best_slot} at "
          f"{real.min():,.0f}, against {plan.cost:,.0f} on a free clock: the calendar costs "
          f"it {real.min() - plan.cost:,.0f}")

In [ ]:
_try("exercise 6", _check_slot_costs)

In [ ]:
def _calendar_price() -> None:
    free = sum(best_intervention_time(fu.forecast, CANDIDATES, SCENARIO).cost for fu in FLEET)
    slotted = sum(float(slot_costs(fu.forecast, SLOTS, SCENARIO).min()) for fu in FLEET)
    print(f"free clock, one machine at a time      {free:>12,.0f}")
    print(f"best slot each, capacity ignored       {slotted:>12,.0f}")
    print(f"the calendar alone costs               {slotted - free:>12,.0f}  "
          f"({100 * (slotted - free) / free:.1f}%)")
    print("\nand nobody has said yet that two of these machines want the same Thursday.")


_try("what the calendar costs", _calendar_price, needs=("exercise 3", "exercise 4", "exercise 6"))

## 10. Exercise 7 — `fleet_cost_matrix()`

One row per machine, one column per slot. This is the object the rest of the module argues
over, and it is the object to put in front of a maintenance planner: not a list of RULs, not
a list of alarms, but the price of every choice they could make this fortnight.

A machine with no finite entry in its row is a machine the calendar cannot serve. That is a
planning problem — hire a contractor, borrow a slot, accept the failure — and not something
to round away here.

<details><summary>💡 Hint 1 — what to think about</summary>

Which way round will the schedulers read this matrix: machines down the rows, or slots? A
square test case cannot tell you, so decide deliberately. Then separate two situations: a
row that CONTAINS an unreachable slot, which is normal, and a row with no reachable slot
at all, which is a machine the calendar cannot serve. Which of those must raise?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Raise `ValueError` on an empty fleet or an empty calendar. Build one row per forecast with
your `slot_costs` and stack the rows, so the shape is machines by slots. Then inspect
every row: one with no finite entry raises `ValueError` naming the machine, rather than
being dropped (which renumbers every machine below it) or zeroed.
</details>

In [ ]:
def fleet_cost_matrix(forecasts, slots, prices: Prices) -> np.ndarray:
    """Expected cost of every (machine, slot) pair, as a (n_machines, n_slots) float array.

    Row `i` is `slot_costs(forecasts[i], slots, prices)`. Raise `ValueError` if `forecasts` or
    `slots` is empty, or if any row is entirely non-finite — a machine with no feasible slot
    is a planning problem and must not be silently dropped or zeroed.

    Returns the matrix.

    Example:
        >>> f = RULForecast(0, 0, np.array([0, 1, 2, 3]), np.array([0.1, 0.4, 0.4, 0.1]))
        >>> p = Prices(planned=10.0, unplanned=100.0, life_per_hour=1.0)
        >>> fleet_cost_matrix([f, f], [0, 2], p).round(2)
        array([[20.5, 91.1],
               [20.5, 91.1]])
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_fleet_cost_matrix() -> None:
    f = RULForecast(0, 0, np.array([0, 1, 2, 3]), np.array([0.1, 0.4, 0.4, 0.1]))
    cheap = Prices(planned=10.0, unplanned=100.0, life_per_hour=1.0)
    got = fleet_cost_matrix([f, f], [0, 2], cheap)
    assert got.shape == (2, 2), f"expected shape (2, 2) — machines by slots — got {got.shape}"
    assert np.allclose(got, [[20.5, 91.1], [20.5, 91.1]]), (
        f"expected [[20.5, 91.1], [20.5, 91.1]], got {np.round(got, 3).tolist()}. A "
        "transposed matrix here quietly schedules slots onto machines for the rest of the "
        "notebook"
    )
    late = RULForecast(1, 500, np.array([0, 1]), np.array([0.5, 0.5]))
    try:
        fleet_cost_matrix([f, late], [0, 2], cheap)
    except ValueError:
        pass
    else:
        raise AssertionError("a machine whose every slot is in its own past has no feasible "
                             "plan and must raise ValueError rather than be dropped")
    later = RULForecast(2, 100, np.array([0, 1, 2, 3]), np.array([0.1, 0.4, 0.4, 0.1]))
    mixed = fleet_cost_matrix([f, later], [0, 100, 102], cheap)
    assert math.isinf(mixed[1, 0]) and np.all(np.isfinite(mixed[1, 1:])), (
        f"the second forecast was made at hour 100, so slot 0 is in its past and the other "
        f"two are not; expected [inf, 20.5, 91.1], got {np.round(mixed[1], 3).tolist()}. A "
        "row that merely CONTAINS an inf is a normal row — only a row with no finite entry "
        "at all is a machine the calendar cannot serve"
    )
    for bad in (([], [0, 2]), ([f], [])):
        try:
            fleet_cost_matrix(bad[0], bad[1], cheap)
        except ValueError:
            continue
        raise AssertionError("an empty fleet or an empty calendar must raise ValueError")
    real = fleet_cost_matrix([fu.forecast for fu in FLEET], SLOTS, SCENARIO)
    assert real.shape == (len(FLEET), SLOTS.size), (
        f"on the lesson's own fleet the matrix must be {len(FLEET)} machines by "
        f"{SLOTS.size} slots; got {real.shape}. A transposed shape here means rows and "
        "columns are the wrong way round for every scheduler below"
    )
    print(f"exercise 7 looks right — {real.shape[0]} machines x {real.shape[1]} slots, "
          f"cheapest single choice {real.min():,.0f}, dearest {real.max():,.0f}")
    print("each machine's own favourite slot: "
          f"{[int(SLOTS[j]) for j in real.argmin(axis=1)]}")

In [ ]:
_try("exercise 7", _check_fleet_cost_matrix)

## 11. Exercise 8 — `greedy_schedule()`: what the planner actually does

Look at the list of favourite slots the last cell printed. Some of them are the same slot,
and the workshop takes one machine at a time.

The planner works down the list. Each machine gets the cheapest slot still free when its
turn comes; whoever is further down the list takes what is left. It is first come, first
served, it is defensible, it is what a spreadsheet does — and exercise 9 will measure what
it costs.

<details><summary>💡 Hint 1 — what to think about</summary>

This is the planner's method, not the optimum: each machine takes its favourite without
asking whether a later machine needs it more, and the machines are taken in row order, not
sorted by anything. As you go down the list, what do you need to remember about each slot?
How is a tie between two equally cheap free slots broken? And what happens to a machine
for which every slot still free is infinite?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Refuse a capacity below one, and more machines than the slots can hold in total, with
`ValueError`. Keep a count of the room left in each slot. For each row in turn, consider
only the slots with room left and take the cheapest, the lowest index on a tie, even when
all of them are infinite; use up one unit of its room and add its entry to the total.
Return the choices as a tuple of ints together with the total.
</details>

In [ ]:
def greedy_schedule(costs: np.ndarray, capacity: int = CAPACITY) -> Schedule:
    """First come, first served: machine by machine, cheapest slot with room left.

    Work through the rows of `costs` in order. Give each machine the cheapest slot that still
    has spare capacity, breaking a tie towards the LOWEST slot index, and use up one unit of
    that slot's capacity. Raise `ValueError` if `capacity` is less than 1, or if there is not
    enough capacity in total for every machine.

    Returns a `Schedule(assignment, total)` where `assignment` is a tuple of slot indices, one
    per machine in row order, and `total` is the sum of the chosen entries.

    Example:
        >>> c = np.array([[1.0, 2.0], [1.0, 5.0]])
        >>> greedy_schedule(c, capacity=1)
        Schedule(assignment=(0, 1), total=6.0)
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_greedy_schedule() -> None:
    two = np.array([[1.0, 2.0], [1.0, 5.0]])
    got = greedy_schedule(two, capacity=1)
    assert got.assignment == (0, 1) and got.total == 6.0, (
        f"first come first served: machine 0 takes slot 0 (its cheapest) and machine 1 is "
        f"left with slot 1, for 6.0. Got {got}. An assignment of (1, 0) totalling 3.0 is the "
        "OPTIMAL answer, not the greedy one — that is exercise 9"
    )
    ties = np.array([[3.0, 3.0, 9.0], [3.0, 9.0, 9.0]])
    tied = greedy_schedule(ties, capacity=1)
    assert tied.assignment == (0, 1), (
        f"machine 0 sees a tie between slots 0 and 1 and must take the LOWEST index; got "
        f"{tied.assignment}"
    )
    roomy = np.array([[1.0, 7.0], [2.0, 7.0], [3.0, 7.0]])
    wide = greedy_schedule(roomy, capacity=2)
    assert wide.assignment == (0, 0, 1) and abs(wide.total - 10.0) < 1e-9, (
        f"with capacity 2 the first two machines share slot 0 and the third is pushed out; "
        f"expected (0, 0, 1) totalling 10.0, got {wide}"
    )
    for bad_cap in (0, -1):
        try:
            greedy_schedule(two, capacity=bad_cap)
        except ValueError:
            continue
        raise AssertionError(f"capacity={bad_cap} must raise ValueError")
    try:
        greedy_schedule(np.ones((3, 2)), capacity=1)
    except ValueError:
        pass
    else:
        raise AssertionError("3 machines into 2 slots at capacity 1 does not fit, and must "
                             "raise ValueError rather than silently drop one")
    starved = greedy_schedule(np.array([[1.0, math.inf], [math.inf, math.inf]]), capacity=1)
    assert starved.assignment == (0, 1) and math.isinf(starved.total), (
        f"machine 0 takes the only finite slot and machine 1 must still be given the slot "
        f"that is left, infinite though it is: expected ((0, 1), inf), got {starved}. If this "
        "crashed, the inner search never allowed for every remaining slot being infinite"
    )
    matrix = fleet_cost_matrix([fu.forecast for fu in FLEET], SLOTS, SCENARIO)
    plan = greedy_schedule(matrix, CAPACITY)
    assert len(set(plan.assignment)) == len(plan.assignment), (
        f"at capacity 1 no slot may appear twice; got {plan.assignment}"
    )
    print(f"exercise 8 looks right — the planner's schedule is "
          f"{[int(SLOTS[j]) for j in plan.assignment]} costing {plan.total:,.0f}")

In [ ]:
_try("exercise 8", _check_greedy_schedule)

## 12. Exercise 9 — `best_schedule()`, and the price of a full workshop

The cheapest feasible assignment, not the cheapest-looking one machine at a time. At this
size you can simply try every assignment that respects the capacity — this fleet and this
shortlist leave few enough of them to enumerate outright. Recursion over the machines,
pruning a branch as soon as it is already dearer than the best complete answer, does it in
milliseconds; so does `itertools.product` with a capacity filter. The grader cares about the
answer, not the route.

Ties go to the **lexicographically smallest** assignment, so the function returns one
definite schedule rather than whichever one the iteration order happened to reach first.

<details><summary>💡 Hint 1 — what to think about</summary>

The optimum may ask a machine to give up its favourite slot so another machine has
somewhere cheap to go, which greedy never does. What makes an assignment feasible when
capacity is two rather than one: is the limit per slot, per machine, or on the whole
schedule? And among schedules that cost the same, which one does the docstring ask for,
whatever order your search happens to visit them in?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Validate exactly as `greedy_schedule` does. Search the assignments machine by machine,
trying slots in increasing index order and only those with room left, and keep a running
total. Abandon a branch as soon as its partial total is already strictly dearer than the
best complete schedule so far; costs are never negative, so it cannot recover. Replace the
best only when a complete schedule is strictly cheaper: with slots tried in index order,
that keeps the lexicographically smallest on a tie. If every feasible schedule is
infinite, still return one, with an infinite total.
</details>

In [ ]:
def best_schedule(costs: np.ndarray, capacity: int = CAPACITY) -> Schedule:
    """The cheapest assignment of machines to slots that respects the workshop's capacity.

    Every machine gets exactly one slot; no slot takes more than `capacity` machines.
    Minimise the total of the chosen entries; they are expected costs and so are never
    negative, which means a partial assignment already dearer than a complete answer can be
    abandoned rather than finished. Among equal-cost schedules return the
    lexicographically smallest `assignment` tuple. Validate `capacity` and total capacity
    exactly as `greedy_schedule` does.

    Returns a `Schedule(assignment, total)`.

    Example:
        >>> c = np.array([[1.0, 2.0], [1.0, 5.0]])
        >>> best_schedule(c, capacity=1)
        Schedule(assignment=(1, 0), total=3.0)
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_best_schedule() -> None:
    two = np.array([[1.0, 2.0], [1.0, 5.0]])
    got = best_schedule(two, capacity=1)
    assert got.assignment == (1, 0) and abs(got.total - 3.0) < 1e-9, (
        f"expected Schedule((1, 0), 3.0), got {got}. Machine 0 gives up its favourite slot "
        "because machine 1 has nowhere else cheap to go — that swap is the entire difference "
        "between this function and exercise 8"
    )
    tie = np.array([[1.0, 1.0], [1.0, 1.0]])
    tied = best_schedule(tie, capacity=1)
    assert tied.assignment == (0, 1), (
        f"every feasible schedule here costs 2.0, so return the lexicographically smallest "
        f"assignment (0, 1); got {tied.assignment}"
    )
    roomy = np.array([[1.0, 7.0], [2.0, 7.0], [3.0, 7.0]])
    wide = best_schedule(roomy, capacity=2)
    assert wide.assignment == (0, 0, 1) and abs(wide.total - 10.0) < 1e-9, (
        f"with capacity 2 two machines may share slot 0; expected (0, 0, 1) totalling 10.0, "
        f"got {wide}. An answer of (0, 1, 1) means capacity is being applied to the wrong "
        "axis"
    )
    infeasible = np.array([[math.inf, 4.0], [math.inf, 1.0]])
    forced = best_schedule(infeasible, capacity=1)
    assert forced.assignment in ((0, 1), (1, 0)) and math.isinf(forced.total), (
        f"when one machine can only be served by a slot the other must have, the total is "
        f"inf and the caller is told so; got {forced}"
    )
    for bad_cap in (0, -3):
        try:
            best_schedule(two, capacity=bad_cap)
        except ValueError:
            continue
        raise AssertionError(f"capacity={bad_cap} must raise ValueError")
    try:
        best_schedule(np.ones((5, 2)), capacity=2)
    except ValueError:
        pass
    else:
        raise AssertionError("5 machines into 2 slots at capacity 2 does not fit")
    matrix = fleet_cost_matrix([fu.forecast for fu in FLEET], SLOTS, SCENARIO)
    opt = best_schedule(matrix, CAPACITY)
    grd = greedy_schedule(matrix, CAPACITY)
    assert opt.total <= grd.total + 1e-9, (
        f"the optimum ({opt.total:,.0f}) can never be dearer than the greedy schedule "
        f"({grd.total:,.0f})"
    )
    print(f"exercise 9 looks right — the optimal schedule is "
          f"{[int(SLOTS[j]) for j in opt.assignment]} costing {opt.total:,.0f}")

In [ ]:
_try("exercise 9", _check_best_schedule)

In [ ]:
# The schedules below are built from YOUR cost matrix, which is built from YOUR
# slot_costs, which prices with YOUR expected_cost_at.
_FOR_SCHEDULES = ("exercise 3", "exercise 6", "exercise 7", "exercise 8", "exercise 9")


def _price_the_workshop() -> None:
    matrix = fleet_cost_matrix([fu.forecast for fu in FLEET], SLOTS, SCENARIO)
    free = sum(best_intervention_time(fu.forecast, CANDIDATES, SCENARIO).cost for fu in FLEET)
    uncapped = float(matrix.min(axis=1).sum())
    grd = greedy_schedule(matrix, CAPACITY)
    opt = best_schedule(matrix, CAPACITY)
    print(f"{'a free clock, one machine at a time':<44}{free:>12,.0f}  (impossible)")
    print(f"{'best slot each, capacity ignored':<44}{uncapped:>12,.0f}  (impossible)")
    print(f"{'first come first served, one per slot':<44}{grd.total:>12,.0f}  (feasible)")
    print(f"{'cheapest feasible schedule':<44}{opt.total:>12,.0f}  (feasible)")
    print()
    print(f"the calendar costs            {uncapped - free:>12,.0f}")
    print(f"the capacity limit costs      {opt.total - uncapped:>12,.0f}")
    print(f"the planner's heuristic costs {grd.total - opt.total:>12,.0f}")
    assert opt.total > uncapped, (
        "on this fleet the capacity limit must bite: the machines want overlapping slots"
    )
    assert grd.total > opt.total, (
        f"first come first served should be measurably worse than the optimum here "
        f"({grd.total:,.0f} against {opt.total:,.0f})"
    )
    moved = [i for i in range(len(FLEET))
             if grd.assignment[i] != opt.assignment[i]]
    print(f"\n{len(moved)} of {len(FLEET)} machines sit in a different slot under the two "
          f"schedules: units {moved}.\nThe capacity price is real and unavoidable. The "
          f"planner's is neither, and it is the one\nnobody measures, because a "
          f"first-come-first-served plan looks locally sensible at every step.")


_try("what the workshop costs", _price_the_workshop, needs=_FOR_SCHEDULES + ("exercise 4",))

In [ ]:
def _plot_schedules() -> None:
    matrix = fleet_cost_matrix([fu.forecast for fu in FLEET], SLOTS, SCENARIO)
    grd = greedy_schedule(matrix, CAPACITY)
    opt = best_schedule(matrix, CAPACITY)
    fig, ax = plt.subplots(figsize=(7.2, 3.4))
    im = ax.imshow(matrix / 1e3, cmap="YlOrRd", aspect="auto")
    for i in range(matrix.shape[0]):
        ax.plot([grd.assignment[i]], [i], "s", ms=13, mfc="none", mec="black", mew=1.4)
        ax.plot([opt.assignment[i]], [i], "o", ms=7, color="navy")
    ax.set_xticks(range(SLOTS.size))
    ax.set_xticklabels([str(int(s)) for s in SLOTS])
    ax.set_yticks(range(len(FLEET)))
    ax.set_yticklabels([f"unit {fu.forecast.unit}" for fu in FLEET])
    ax.set_xlabel("calendar slot, absolute hour")
    ax.set_title("expected cost, thousands · square = first come first served, dot = optimal")
    fig.colorbar(im, ax=ax)
    _show(fig)
    print("plotted: where the two schedules disagree, and what the disagreement costs")


_try("schedule plot", _plot_schedules, needs=_FOR_SCHEDULES)

## 13. Common mistakes

- **Planning at the mean RUL.** Section 6 priced it on every machine in this fleet. The mean
  is the number the dashboard shows and it is not the answer to any decision.
- **Planning at a fixed percentile.** P10 is a better habit than the mean, and it is still
  not the answer: the right quantile depends on the three prices, and section 7's `t*/mean`
  column shows it is not even a constant fraction of the mean.
- **Dividing the hazard by `survival[k]`.** The population at risk at the start of hour `k`
  includes the machines that fail during it. Off by one bin, and infinite at the tail.
- **Treating a failure at exactly the booked hour as a catch.** It is a failure, and mixing
  the convention between the expected and the realised cost functions puts a whole bin's
  probability on the wrong side of the ledger.
- **Letting a slot in the past cost 0.** An unreachable option priced at zero wins every
  `min` in the module.
- **Believing a narrow forecast because it is narrow.** Section 8 priced that.
- **Scheduling first come, first served.** Section 12 priced that too.
- **Reporting the capacity price and the planner's error as one number.** One of them is a
  fact about the workshop and the other is a fact about the planning method. Only the second
  one is free to fix.

In [ ]:
def plan_at_percentile(forecast: RULForecast, q: float) -> int:
    """The 'pull it at the Qth percentile' rule of thumb, written out. Do not copy it."""
    pmf = np.asarray(forecast.pmf, dtype=float)
    idx = int(np.searchsorted(np.cumsum(pmf), q - 1e-12, side="left"))
    return int(forecast.hours[min(idx, forecast.hours.size - 1)])


def _price_the_rules_of_thumb() -> None:
    print(f"{'unit':>4} {'optimal':>8} {'at mean':>8} {'at P10':>8} {'at P25':>8} "
          f"{'| cost(opt)':>11} {'cost(mean)':>11} {'cost(P10)':>11} {'cost(P25)':>11}")
    best_rule = {"mean": 0.0, "P10": 0.0, "P25": 0.0}
    for fu in FLEET:
        f = fu.forecast
        s = forecast_summary(f)
        plan = best_intervention_time(f, CANDIDATES, SCENARIO)
        rules = {"mean": int(round(s.mean)), "P10": plan_at_percentile(f, 0.10),
                 "P25": plan_at_percentile(f, 0.25)}
        costs = {k: expected_cost_at(f, v, SCENARIO) for k, v in rules.items()}
        for k in best_rule:
            best_rule[k] = max(best_rule[k], costs[k] / plan.cost)
        print(f"{f.unit:>4} {plan.hours_ahead:>8} {rules['mean']:>8} {rules['P10']:>8} "
              f"{rules['P25']:>8} {plan.cost:>11,.0f} {costs['mean']:>11,.0f} "
              f"{costs['P10']:>11,.0f} {costs['P25']:>11,.0f}")
    print(f"\nworst overspend on this fleet: at the mean {best_rule['mean']:.2f}x, at P10 "
          f"{best_rule['P10']:.2f}x, at P25 {best_rule['P25']:.2f}x.")
    print("A percentile rule is much better than the mean and is still a guess at the prices."
          "\nThe prices are known. Use them.")


_try("rules of thumb, priced", _price_the_rules_of_thumb, needs=_FOR_PLAN)

## 14. Self-check

1. Two machines have the same mean RUL of 300 h; one forecast's sd is 30 h and the other's
   is 150 h. Against the same prices, the wider forecast's optimal intervention is:
   - (a) later, because a vague forecast is a reason to wait for more evidence
   - (b) the same, because the expected remaining life is the same
   - (c) earlier, because its near tail carries real probability and so reaches the
         balancing hazard sooner

2. A vendor offers an RUL model 30% better on RMSE that reports no uncertainty. For the
   decision in this module it is:
   - (a) unusable as it stands: the intervention time is set by the shape of the
         distribution, and a point estimate has no shape
   - (b) a clear win, since a better point estimate is a better plan
   - (c) equivalent, since you can assume a fixed spread around the point estimate

3. Your tight forecast said 320 h, the machine stopped at 190, and you paid the unplanned
   price — yet its *expected* cost was the lowest on the fleet. The right reading is:
   - (a) bad luck; the plan was optimal given the belief
   - (b) bad luck AND a warning: an expected cost is only as good as the width's
         calibration, so the width needs auditing against outcomes, not just the centre
   - (c) the cost model is wrong, because it produced a plan that failed

4. Capacity raised the fleet total by one measured amount; switching from first come, first
   served to the cheapest feasible schedule lowered it by another. Those two numbers are:
   - (a) a constraint you cannot avoid, and an avoidable loss from how you respected it
   - (b) the same quantity measured twice
   - (c) both arguments for relaxing the capacity limit

5. A planner wants one number per machine so they can sort the list. The honest answer is:
   - (a) the mean RUL, since it is the expected value
   - (b) the P10, since it is conservative
   - (c) the intervention hour AND what the second-best slot would cost, because the sort
         is only useful when it knows what giving way is worth

Answers, with the reasoning, are in this lesson's worked solution in the course repository.

In [ ]:
# The deliverable. A plan with no prices attached is a date somebody will move, and a date
# moved without re-pricing is a decision made by whoever moved it.
def _handover() -> None:
    matrix = fleet_cost_matrix([fu.forecast for fu in FLEET], SLOTS, SCENARIO)
    opt = best_schedule(matrix, CAPACITY)
    grd = greedy_schedule(matrix, CAPACITY)
    uncapped = float(matrix.min(axis=1).sum())
    print("INTERVENTION PLAN — ships with the prices, or it does not ship")
    print(f"  prices             planned {SCENARIO.planned:,.0f} · unplanned "
          f"{SCENARIO.unplanned:,.0f} · {SCENARIO.life_per_hour:,.0f} per hour of life")
    print(f"  balance hazard     "
          f"{SCENARIO.life_per_hour / (SCENARIO.unplanned - SCENARIO.planned):.6f} per hour — "
          f"pull when the hazard reaches this")
    print(f"  calendar           {CALENDAR.size} slots in {CALENDAR_DAYS} days, "
          f"{SLOTS.size} shortlisted, {CAPACITY} machine per slot")
    print(f"  schedule           total {opt.total:,.0f}, against {uncapped:,.0f} if the "
          f"workshop were unlimited")
    print(f"  method             exhaustive over feasible assignments; first come first "
          f"served would cost {grd.total - opt.total:,.0f} more")
    for i, fu in enumerate(FLEET):
        s = forecast_summary(fu.forecast)
        free_plan = best_intervention_time(fu.forecast, CANDIDATES, SCENARIO)
        row = matrix[i]
        second = float(np.sort(row)[1])
        print(f"    unit {fu.forecast.unit}  slot hour {int(SLOTS[opt.assignment[i]]):>4}  "
              f"cost {row[opt.assignment[i]]:>9,.0f}  "
              f"(free-clock optimum {free_plan.hours_ahead:>3} h, mean RUL {s.mean:>5.0f} h, "
              f"P10 {s.p10:>3} h; giving way to the next-best slot is worth "
              f"{second - row.min():,.0f})")
    print("  re-derive whenever a price changes, a slot moves, or module 5 re-fits. All three "
          "are\n  changes to the decision, not to the model.")


_try("handover note", _handover, needs=_FOR_PLAN + _FOR_SCHEDULES)

## What you built, and where it goes next

Module 1 asked what a threshold costs. This module asked the same question of a
distribution and got a different kind of answer: not a threshold at all, but a *time*, set
by the hazard rate rather than by the centre of the forecast, and then dragged off that time
by a calendar and by a workshop that takes one machine at a time.

Three numbers came out of it, and they are three different kinds of thing. The calendar
price is a fact about the working week. The capacity price is a fact about the workshop, and
it is the number to put in front of whoever decides how many bays it has. The planner's
error is a fact about the method, it is free to fix, and it is the only one of the three
that nobody was measuring.

Module 7 puts this decision on an OT network, where the forecast has to reach the planner at
all. Module 8 comes back to the prices, because the base rate that sets them drifts, and a
balance hazard derived from last year's prices is last year's plan.

The habit to carry forward: **when somebody hands you an RUL, ask for its width, and when
somebody hands you a date, ask what the next-best date would have cost.**

In [ ]:
_MARKS = {"passed": "✅", "failed": "❌", "not started": "⏳"}


def _progress_board() -> None:
    """One line per exercise, from the latest run of its check, then the tally."""
    width = max(len(", ".join(funcs)) for funcs in _EXERCISES.values())
    print("progress board")
    for label, funcs in _EXERCISES.items():
        state = _STATUS.get(label, "not started")
        print(f"  {_MARKS[state]} {label:<12} {', '.join(funcs):<{width}}  {state}")
    done = sum(_STATUS.get(label) == "passed" for label in _EXERCISES)
    print(f"\n{done} of {len(_EXERCISES)} exercises complete")
    failing = [label for label in _EXERCISES if _STATUS.get(label) == "failed"]
    if failing:
        print("failing right now: " + ", ".join(failing) + ". Each one printed what went "
              "wrong in its own cell above, and every exercise has hints you can open.")
    elif done < len(_EXERCISES):
        print("work top to bottom: every exercise has hints you can open above its code.")


# Your progress board. Every check is re-run here, quietly, against your code as it stands
# now — each one already printed its feedback in its own cell above — so the board is
# current even if you edited an exercise and did not re-run its check.
if __name__ == "__main__":
    with contextlib.redirect_stdout(io.StringIO()):
        for _name, _check in (("exercise 1", _check_forecast_summary),
                              ("exercise 2", _check_survival_and_hazard),
                              ("exercise 3", _check_expected_cost_at),
                              ("exercise 4", _check_best_intervention_time),
                              ("exercise 5", _check_realised_cost),
                              ("exercise 6", _check_slot_costs),
                              ("exercise 7", _check_fleet_cost_matrix),
                              ("exercise 8", _check_greedy_schedule),
                              ("exercise 9", _check_best_schedule)):
            _try(_name, _check)
    _progress_board()
    print(f"\nnotebook wall time so far: {time.perf_counter() - _LESSON_T0:.1f}s")
    # A stub you have not reached yet is not a failure. A check that ran and came back wrong
    # is: in a script or under CI it ends this run non-zero, rather than letting a green exit
    # code paper over it. Inside a notebook kernel the board above has already said so, in a
    # line rather than a traceback at the foot of the page.
    if _FAILED_CHECKS and "ipykernel" not in sys.modules:
        raise SystemExit("checks failed: " + ", ".join(dict.fromkeys(_FAILED_CHECKS)))